 - https://github.com/swan-cern/sparkmonitor

# BUG
**After kernel shutdown or restart you must refresh the page with "Ctrl+Shift+R"** 

In [3]:
from sparkmonitor import kernelextension

monitor = kernelextension.monitor

print("Socket port:", monitor.getPort())
print("Frontend connected:", hasattr(monitor, "comm"))

Socket port: 44845
Frontend connected: True


In [4]:
from sparkmonitor import kernelextension
monitor = kernelextension.monitor
if not hasattr(monitor, "comm"):
    print(
        "SparkMonitor frontend is not connected. "
        "Reload the JupyterLab page with CTRL SHIFT R."
    )
    raise SystemExit

In [1]:
!jupyter labextension list

JupyterLab v4.6.2
/opt/conda/share/jupyter/labextensions
        jupyterlab_pygments v0.3.0 enabled OK (python, jupyterlab_pygments)
        sparkmonitor v3.3.0 enabled OK (python, sparkmonitor)
        jupyter-matplotlib v0.12.0 enabled OK
        nbdime-jupyterlab v3.0.4 enabled OK
        @jupyter-notebook/lab-extension v7.6.1 enabled OK
        @jupyter-widgets/jupyterlab-manager v5.0.15 enabled OK (python, jupyterlab_widgets)
        @jupyterlab/git v0.54.0 enabled OK (python, jupyterlab-git-core)


Disabled extensions:
    @jupyterlab/apputils-extension:announcements


In [2]:
!jupyter server extension list

Config dir: /home/jovyan/.jupyter

Config dir: /opt/conda/etc/jupyter
    jupyter_lsp enabled
    - Validating jupyter_lsp...
      jupyter_lsp 2.3.1 OK
    jupyter_server_terminals enabled
    - Validating jupyter_server_terminals...
      jupyter_server_terminals 0.5.4 OK
    jupyterlab enabled
    - Validating jupyterlab...
      jupyterlab 4.6.2 OK
    jupyterlab_git enabled
    - Validating jupyterlab_git...
      jupyterlab_git 0.54.0 OK
    nbclassic enabled
    - Validating nbclassic...
A `_jupyter_server_extension_points` function was not found in nbclassic. Instead, a `_jupyter_server_extension_paths` function was found and will be used for now. This function name will be deprecated in future releases of Jupyter Server.
      nbclassic 1.3.3 OK
    nbdime enabled
    - Validating nbdime...
      nbdime 4.0.4 OK
    notebook enabled
    - Validating notebook...
      notebook 7.6.1 OK
    notebook_shim enabled
    - Validating notebook_shim...
      notebook_shim  OK

Config d

In [3]:
%load_ext sparkmonitor.kernelextension

The sparkmonitor.kernelextension extension is already loaded. To reload it, use:
  %reload_ext sparkmonitor.kernelextension


In [4]:
%%bash
which jupyter
which python
python -m pip show sparkmonitor
jupyter lab --version

/opt/conda/bin/jupyter
/opt/conda/bin/python
Name: sparkmonitor
Version: 3.3.0
Summary: Jupyter Notebook & Lab extension to monitor Apache Spark jobs from a notebook
Home-page: https://github.com/swan-cern/sparkmonitor
Author: 
Author-email: Krishnan R <krishnanr1997@gmail.com>
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "

In [5]:
from sparkmonitor import kernelextension

monitor = kernelextension.monitor

print("Socket port:", monitor.getPort())
print("Frontend connected, should be true:", hasattr(monitor, "comm"))

Socket port: 53939
Frontend connected, should be true: True


In [6]:
%run_nb spark-start

Args: Namespace(data_format='none', port_offset=2) - unknown_args: []
Spark version: 4.1.3, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: None
Spark packages: org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3
Spark extensions: 
Spark catalog configs: {}
spark.sql.shuffle.partitions: 200
spark.sparkContext.master: local[2]


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcatalog
spark_catalog


Version,4.1.3
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        13Gi        37Gi       291Mi        12Gi        48Gi
Swap:          8.0Gi          0B       8.0Gi


In [7]:
print(
    spark.sparkContext.getConf().get(
        "spark.extraListeners",
        "NOT CONFIGURED"
    )
)

print(
    spark.sparkContext.getConf().get(
        "spark.driver.extraClassPath",
        "NOT CONFIGURED"
    )
)

sparkmonitor.listener.JupyterSparkMonitorListener
/opt/conda/lib/python3.13/site-packages/sparkmonitor/listener_spark4_2.13.jar


In [8]:
from pyspark.sql import functions as F

df = (
    spark.range(0, 10_000_000, numPartitions=32)
    .withColumn("group_id", F.col("id") % 100)
)

result = (
    df.groupBy("group_id")
      .count()
      .orderBy("group_id")
      .collect()
)

print(f"Returned {len(result)} rows")

Returned 100 rows


In [9]:
total = (
    spark.sparkContext
         .parallelize(range(1_000_000), 32)
         .map(lambda x: x * 2)
         .filter(lambda x: x % 3 == 0)
         .sum()
)

print(total)

333333666666


In [10]:
from itables import show

show(
    df.limit(1000).toPandas(),
    scrollX=True,
    pageLength=25
)

In [11]:
viewdf(df)